# 03 · LoRA / QLoRA fine-tuning
> **Status:** created locally, syntax-checked only. **NOT yet run on Colab** — update docs/PROGRESS.md after the first real run.

Only after notebook 02 has a measured baseline. Data = manifest `train_ids` (1236), never the 309 held-out IDs. Training is OFF until `RUN_TRAINING = True`.

In [ ]:
# Bootstrap: mount Drive, clone/update repo (token only from Colab Secrets, never hardcoded)
REPO_URL = "https://github.com/Quang1104/lifemate_superklang.git"
BRANCH = "claude/zalo-lifemate-handoff-5c2f7d"
REPO = "/content/lifemate_superklang"
import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
url = REPO_URL
try:
    from google.colab import userdata
    url = REPO_URL.replace("https://", "https://x-access-token:" + userdata.get("GITHUB_TOKEN") + "@")
except Exception:
    pass  # public repo, or secret GITHUB_TOKEN not set
if os.path.isdir(REPO + "/.git"):
    subprocess.run(["git", "-C", REPO, "pull", "-q", "--ff-only", url, BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, REPO], check=True)
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", REPO_URL], check=True)
os.chdir(REPO); sys.path.insert(0, REPO)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -r requirements-colab.txt

In [ ]:
from src.bench import colab_setup as cs, evaluate
cfg = cs.load_config()
print(cs.gpu_report())
DATA_DIR = cs.restore_dataset(cfg)          # unzip Drive archive -> runtime disk (skipped if present)
RESULTS = cs.results_dir(cfg)                # Drive: /MyDrive/lifemate/results
GOLD = evaluate.ensure_gold(DATA_DIR, RESULTS / "gold_validation.jsonl")  # TRAIN held-out hashes, CPU
print(DATA_DIR, GOLD)

In [ ]:
from src.bench import sft_data, finetune
SFT = RESULTS / "sft_train.jsonl"
if not SFT.exists():
    print(sft_data.build(DATA_DIR, SFT))       # CPU, one ToolEnv per sample, gold replay
profile = finetune.vram_profile(); profile

In [ ]:
RUN_TRAINING = False
MAX_STEPS = 200                                # start short; resume continues from the last checkpoint
OUT = RESULTS.parent / "checkpoints" / "lora_v1"
if RUN_TRAINING:
    adapter = finetune.train(cs.fetch_model(cfg, "planner"), SFT, OUT, max_steps=MAX_STEPS, profile=profile)
    print("adapter:", adapter)

Evaluate the adapter with notebook 02 (`ADAPTER = OUT / 'adapter'`) on the same IDs, then compare in 04.